# PV on/off study — train the two models (fixed-topology, isolates the PV effect)

Trains the deployed model configuration (coupled multi-task CSGNNv2 (alpha=1, the deployed recipe)) on
the SAME 27-bus grid, identical recipe, 5 seeds each — only the PV active output differs:

- **PV-OFF** = 27-bus, Bus 27 present, **PV active output = 0** (`zero`)
- **PV-ON**  = 27-bus, Bus 27 present, **PV = 100 MW** (`on`)

Same 27-node / 48-branch input structure both ways, so any difference is the PV injection.

**Upload to one Google Drive folder:** `boost_core.py`, `data_pv_zero.npz`,
`data_pv_zero_split.npz`, `data_pv_on.npz`, `data_pv_on_split.npz`.
**Runtime -> GPU.** Then Run all. Saves `pv_zero.pt`, `pv_on.pt`, `pv_results.json` back to the folder.

In [ ]:
from google.colab import drive
import glob, os, sys
drive.mount('/content/drive')
# glob the FOLDER NAME (not a data file -- an old copy of the npz elsewhere on Drive would
# mislead FOLDER onto a stale unmasked boost_core.py, which silently produces pre-mask results)
c = glob.glob('/content/drive/MyDrive/**/smart_load_shield_boost', recursive=True)
FOLDER = c[0] if c else '/content/drive/MyDrive/smart_load_shield_boost'
sys.path.insert(0, FOLDER)
need = ['data_pv_zero.npz', 'data_pv_zero_split.npz', 'data_pv_on.npz', 'data_pv_on_split.npz', 'boost_core.py']
missing = [f for f in need if not os.path.exists(os.path.join(FOLDER, f))]
assert not missing, 'missing in ' + FOLDER + ': ' + str(missing)
assert 'base_mask' in open(os.path.join(FOLDER, 'boost_core.py')).read(), \
    'boost_core.py in FOLDER is the OLD unmasked version -- replace it with the round-6 masked one'
print('FOLDER =', FOLDER, '| masked boost_core OK')

In [ ]:
import numpy as np, torch, json
import boost_core as B
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device', DEV)
SEEDS = [0, 1, 2, 3, 4]


@torch.no_grad()
def eval_full(net, d, idx):
    net.eval(); P = []
    for s in range(0, len(idx), 4096):
        b = idx[s:s + 4096]
        r, vm, vu, dv, vb = net(d['NF'][b], d['DENSE'][b], d['ADJ'][b])
        P.append(r.argmax(1))
    P = torch.cat(P); R = d['Tr'][idx]
    acc = (P == R).float().mean().item()
    fs = (((R == 2) & (P == 0)).sum() / (R == 2).sum().clamp(min=1)).item()
    cm = torch.zeros(3, 3, dtype=torch.long)
    for t, p in zip(R.cpu(), P.cpu()): cm[t, p] += 1
    f1 = []
    for k in range(3):
        tp = cm[k, k].item(); fp = cm[:, k].sum().item() - tp; fn = cm[k].sum().item() - tp
        pr = tp / max(1, tp + fp); rc = tp / max(1, tp + fn); f1.append(2 * pr * rc / max(1e-9, pr + rc))
    return dict(acc=acc, false_safe=fs, macro_f1=float(np.mean(f1)), cm=cm.tolist())


def run_mode(mode):
    d = B.build_data(FOLDER + '/data_pv_' + mode + '.npz', FOLDER + '/data_pv_' + mode + '_split.npz', DEV)
    res = []; best_val = -1.0; best_state = None
    for s in SEEDS:
        te, model = B.train_eval(d, seed=s, alpha=1.0, branched=False, stack=False, vbus=False,
                                 epochs=150, amp=(DEV == 'cuda'))
        full = eval_full(model, d, d['idx_te']); full['vmin_mae'] = te['vmin_mae']; res.append(full)
        val = B.evaluate(model, d, d['idx_va'])['acc']
        print('  %s seed%d: acc %.2f%%  false-safe %.3f%%  F1 %.2f%%  Vmin-MAE %.4f'
              % (mode, s, full['acc'] * 100, full['false_safe'] * 100, full['macro_f1'] * 100, te['vmin_mae']))
        if val > best_val:
            best_val = val; best_state = {k: v.clone() for k, v in model.state_dict().items()}
    torch.save(best_state, FOLDER + '/pv_' + mode + '.pt')
    accs = [r['acc'] for r in res]; fss = [r['false_safe'] for r in res]
    f1s = [r['macro_f1'] for r in res]; vmaes = [r['vmin_mae'] for r in res]
    print('%s: acc %.2f+-%.2f%%  false-safe %.3f%%  F1 %.2f%%  Vmin-MAE %.4f  (n_test=%d)\n'
          % (mode.upper(), np.mean(accs) * 100, np.std(accs) * 100, np.mean(fss) * 100,
             np.mean(f1s) * 100, np.mean(vmaes), len(d['idx_te'])))
    return dict(mode=mode, n_bus=int(d['NF'].shape[1]), n_test=len(d['idx_te']),
                acc=float(np.mean(accs)), acc_std=float(np.std(accs)), false_safe=float(np.mean(fss)),
                macro_f1=float(np.mean(f1s)), vmin_mae=float(np.mean(vmaes)), seeds=res)

In [ ]:
out = {}
for mode in ['zero', 'on']:
    print('===== PV ' + mode.upper() + ' =====')
    out[mode] = run_mode(mode)
json.dump(out, open(FOLDER + '/pv_results.json', 'w'), indent=2)
print('saved pv_results.json + pv_zero.pt + pv_on.pt to', FOLDER)
print('\n=== SUMMARY (PV=0 vs PV=100, same 27-bus, mean of 5 seeds) ===')
for m in ['zero', 'on']:
    r = out[m]
    print('PV %-3s (%d-bus)  acc %.2f%%  false-safe %.3f%%  F1 %.2f%%  Vmin-MAE %.4f  (test %d)'
          % (m, r['n_bus'], r['acc'] * 100, r['false_safe'] * 100, r['macro_f1'] * 100, r['vmin_mae'], r['n_test']))